In [ ]:
from pandas_plink import read_plink
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import math

import os
import csv
import json
import gc
import h5py

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, Callback

from scipy import stats
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from phenotype_levels import phenotype_levels_used

In [ ]:
fam_path = "/storage/plzen4-ntis/home/tadsova/rice_data/base_filtered_v0.7_0.8_10kb_1_0.8_50_1.fam"
pheno_path = "phenotypes/all_phenotypes.txt"
prefix = "/storage/plzen4-ntis/home/tadsova/rice_data/base_filtered_v0.7_0.8_10kb_1_0.8_50_1"

In [ ]:
def snp_matrix(prefix):
    """
    Reads PLINK files and converts them into a genotype matrix.
    Handles missing values by imputing them with the most frequent allele 
    for each SNP and returns a compressed int8 matrix.
    """
    (bim, fam, G) = read_plink(prefix, verbose=False)

    X = G.compute().T.astype(np.float32) # sample x snp
    missing_count = np.isnan(X).sum()
    print(f"Celkem chybějících genotypů: {missing_count}")

    # nahrazení NaN za modus
    if missing_count > 0:
        modes = stats.mode(X, axis=0, nan_policy='omit', keepdims=True).mode[0]
        modes = np.nan_to_num(modes, nan=0)
        
        inds = np.where(np.isnan(X))
        X[inds] = np.take(modes, inds[1])
    
    X = X.astype(np.int8)
    print("Genotype matrix shape:", X.shape)

    return X

def get_snp_matrix(prefix, cache_path="snp_matrix/snp_matrix.h5"):
    """
    Loads the genotype matrix from an HDF5 cache file if it exists.
    Otherwise, processes the PLINK files and saves the result to cache.
    """
    if os.path.exists(cache_path):
        print("Načítání matice")
        with h5py.File(cache_path, 'r') as f:
            X = f['X'][:]
        print("Genotype matrix shape:", X.shape)
    else:
        X = snp_matrix(prefix)
        with h5py.File(cache_path, 'w') as f:
            f.create_dataset('X', data=X, compression='gzip', compression_opts=4, chunks=True)
    return X

def prepare_phenotype_data(pheno_path, fam_path, phenotype):
    """
    Merges phenotype data with the .fam file.
    Ensures that only samples present in both files and having a non-null 
    phenotype value are kept.
    """
    pheno = pd.read_csv(pheno_path, sep="\t")
    fam = pd.read_csv(
        fam_path, 
        sep=r"\s+", 
        header=None, 
        names=["FID", "IID", "father", "mother", "sex", "trait"]
    )
    
    pheno_subset = pheno[["3K_DNA_IRIS_UNIQUE_ID", phenotype]].rename(
        columns={"3K_DNA_IRIS_UNIQUE_ID": "IID", phenotype: "VALUE"}
    )
    
    fam["IDX"] = range(len(fam))

    merged_data = fam[["IID", "IDX"]].merge(pheno_subset, on="IID").dropna(subset=["VALUE"])

    merged_data = merged_data[["IDX", "IID", "VALUE"]]
    merged_data["VALUE"] = merged_data["VALUE"].astype(int)

    print(f"Počet vzorků '{phenotype}' po filtraci: {len(merged_data)}")
    
    return merged_data

def analyze_phenotype_distribution(df, phenotype_name):
    """
    Calculates and prints the frequency of each phenotype class.
    Returns the apriori accuracy.
    """
    print(f"\nDistribuce hodnot pro: {phenotype_name}")

    counts = df["VALUE"].value_counts().sort_index()
    print(counts)
    
    total = len(df)
    majority_count = counts.max()
    apriori_acc = majority_count / total
    
    print(f"Apriori accuracy (majority class): {apriori_acc:.4f}")
    
    return apriori_acc

def onehot_encode_phenotype(df, levels, prefix):
    """
    Performs one-hot encoding for the phenotype values based on predefined levels.
    """
    onehot_data = pd.get_dummies(df["VALUE"], prefix=prefix, dtype=np.float32)

    for lvl in levels:
        col = f"{prefix}_{lvl}"
        if col not in onehot_data.columns:
            onehot_data[col] = 0

    column_order = [f"{prefix}_{l}" for l in levels]
    onehot_data = onehot_data[column_order]

    final_df = pd.concat([
        df[["IDX", "IID"]].reset_index(drop=True), 
        onehot_data.reset_index(drop=True)
    ], axis=1)

    print(f'\nOne-hot kódování provedeno.')

    return final_df

def get_clean_labels(df_onehot, levels, prefix):
    """
    Extracts the one-hot encoded values into a clean NumPy array for model training.
    """
    label_cols = []
    for l in levels:
        label_cols.append(prefix + "_" + str(l))
    y = df_onehot[label_cols].values

    return y

def align_genotypes_to_phenotypes(X, df_pheno):
    """
    Filters the full genotype matrix to only include samples that 
    passed phenotype filtering, maintaining the correct order.
    """
    keep_indices = df_pheno["IDX"].values

    X_filtered = X[keep_indices, :]

    return X_filtered

def run_phenotype_pipeline(phenotype_name, X_full_matrix, pheno_path, fam_path):
    """
    The pipeline preparation for a single phenotype:
    1. Loads and filters data.
    2. Analyzes distribution.
    3. Encodes labels.
    4. Aligns genotypes with filtered phenotypes.
    """
    if phenotype_name not in phenotype_levels_used:
        print(f"Fenotyp {phenotype_name} neexistuje.")
        return None
    
    levels = phenotype_levels_used[phenotype_name]
    
    df_pheno = prepare_phenotype_data(pheno_path, fam_path, phenotype_name)
    
    apriori_acc = analyze_phenotype_distribution(df_pheno, phenotype_name)
    
    df_onehot = onehot_encode_phenotype(df_pheno, levels, phenotype_name)
    #print(df_onehot.dtypes)

    y_final = get_clean_labels(df_onehot, levels, phenotype_name)
    
    X_ready_for_onehot = align_genotypes_to_phenotypes(X_full_matrix, df_pheno)

    return X_ready_for_onehot, y_final, levels, apriori_acc

In [ ]:
def select_top_snps(X, n_snps=None, method='entropy'):
    """
    Performs feature selection by ranking SNPs based on their variability or information content.
    Supported methods:
    - 'variance': Selects SNPs with the highest statistical variance.
    - 'entropy': Selects SNPs with the highest entropy (information diversity).
    """
    
    if n_snps is None or n_snps >= X.shape[1]:
        return np.arange(X.shape[1])
    
    X_f = X.astype(np.float32)
    
    if method == 'variance':
        scores = np.var(X_f, axis=0)
    
    elif method == 'entropy':
        scores = np.zeros(X_f.shape[1], dtype=np.float32)
        for val in [0, 1, 2]:
            p = np.mean(X_f == val, axis=0)
            p = np.clip(p, 1e-10, 1)
            scores += -p * np.log2(p)
    
    top_indices = np.argsort(scores)[::-1][:n_snps]
    return top_indices

In [ ]:
def create_train_val_split(X, y, val_size=0.2, random_state=42):
    """
    Splits the dataset into training and validation sets.
    """
    X_train, X_val, y_train, y_val = train_test_split(
        X, 
        y, 
        test_size=val_size, 
        random_state=random_state, 
        shuffle=True
    )

    print("Dokončené rozdělení dat")
    print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
    print(f"X_val:   {X_val.shape},   y_val:   {y_val.shape}")
    
    return X_train, X_val, y_train, y_val

In [ ]:
class GenomicDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, X, y, batch_size=32, mask_prob=0.9, shuffle=True):
        self.X = X
        self.y = y
        self.batch_size = batch_size
        self.mask_prob = mask_prob
        self.shuffle = shuffle
        self.indices = np.arange(len(self.X))
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.X) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, index):
        #změna
        start = index * self.batch_size
        end = min(start + self.batch_size, len(self.indices))
        batch_indices = self.indices[start:end]
        #batch_indices = self.indices[index*self.batch_size:(index+1)*self.batch_size]
        
        X_batch = self.X[batch_indices]
        y_batch = self.y[batch_indices]

        num_samples, num_snps = X_batch.shape
        X_out = np.zeros((num_samples, num_snps, 4), dtype=np.float32)
        
        mask = np.random.rand(num_samples, num_snps) < self.mask_prob

        for val in [0, 1, 2]:
            X_out[:, :, val] = (X_batch == val) & (~mask)
        
        X_out[:, :, 3] = mask
        
        return X_out.reshape(num_samples, -1), y_batch

In [ ]:
def build_genomic_model(input_dim, num_classes, learning_rate=1e-6):
    """
    Builds a simple Softmax Single-layer Neural Network.
    Uses zero initialization for weights to ensure a baseline starting point.
    """
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),

        layers.Dense(num_classes, kernel_initializer='zeros', activation='softmax')
    ])
    
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['categorical_accuracy']
    )
    
    return model

In [ ]:
def plot_history(history):
    """
    Generates plots for Training/Validation Accuracy and Loss over epochs.
    """
    acc = history.history['categorical_accuracy']
    val_acc = history.history['val_categorical_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(acc) + 1)

    plt.figure(figsize=(12, 5))

    # Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(epochs, acc, 'bo-', label='Trénovací acc')
    plt.plot(epochs, val_acc, 'ro-', label='Validační acc')
    plt.title('Přesnost trénování a validace')
    plt.xlabel('Epochs')
    plt.ylabel('Acc')
    plt.legend()

    # Loss
    plt.subplot(1, 2, 2)
    plt.plot(epochs, loss, 'bo-', label='Trénovací loss')
    plt.plot(epochs, val_loss, 'ro-', label='Validační loss')
    plt.title('Ztráta trénování a validace')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
def run_snp_scaling_experiment(X_ready, y_ready, apriori_acc, snp_counts, method, n_repeats, num_epochs, csv_path, learning_rate):
    """
    Automates a full scaling experiment:
    1. Ranks all SNPs.
    2. Iterates through different SNP subset sizes (snp_counts).
    3. Repeats training multiple times to ensure statistical significance.
    4. Logs results (best epoch metrics) incrementally to a CSV file.
    """

    all_indices = select_top_snps(X_ready, n_snps=None, method=method)
    
    csv_columns = ['run', 'n_snps', 'apriori_acc', 'train_acc', 'val_acc', 'train_loss', 'val_loss', 'epochs_run']
    if not os.path.exists(csv_path):
        pd.DataFrame(columns=csv_columns).to_csv(csv_path, index=False)
        print(f"Vytvořen soubor: {csv_path}")
        existing_results = pd.DataFrame(columns=csv_columns)
    else:
        print(f"Přidávám do souboru: {csv_path}")
        existing_results = pd.read_csv(csv_path)
    
    for n in snp_counts:
        label = n if n is not None else X_ready.shape[1]

        idx = all_indices if n is None else all_indices[:n]
        X_sub = X_ready[:, idx]

        if not existing_results.empty and label in existing_results['n_snps'].values:
            last_run = existing_results[existing_results['n_snps'] == label]['run'].max()
        else:
            last_run = 0

        if last_run >= n_repeats:
            print(f"SNP {label}: hotovo ({last_run}/{n_repeats}), skip")
            continue
        
        for run in range(last_run + 1, n_repeats + 1):
            print(f"\n{'='*50}")
            print(f"SNP: {label} | Run {run}/{n_repeats}")
            
            X_train, X_val, y_train, y_val = create_train_val_split(X_sub, y_ready)
            
            print("y_train shape:", y_train.shape)
            print("unique labels:", np.unique(np.argmax(y_train, axis=1)))

            train_gen = GenomicDataGenerator(X_train, y_train, mask_prob=train_mask_probability)
            val_gen = GenomicDataGenerator(X_val, y_val, mask_prob=test_mask_probability, shuffle=False)
            
            model = build_genomic_model(X_sub.shape[1] * 4, y_train.shape[1], learning_rate=learning_rate)
            early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
            
            history = model.fit(
                train_gen,
                validation_data=val_gen,
                epochs=num_epochs,
                callbacks=[early_stop],
                #class_weight=class_weight_dict,
                verbose=1
            )

            #plot_history(history)

            best_idx = history.history['val_loss'].index(min(history.history['val_loss']))
            best_val_acc = history.history['val_categorical_accuracy'][best_idx]
            best_val_loss = history.history['val_loss'][best_idx]
            best_train_acc = history.history['categorical_accuracy'][best_idx]
            best_train_loss = history.history['loss'][best_idx]
            
            print(f"  Train acc: {best_train_acc:.4f} | Val acc: {best_val_acc:.4f} | Apriori: {apriori_acc:.4f}")
            
            row = pd.DataFrame([{
                'run': run,
                'n_snps': label,
                'apriori_acc': apriori_acc,
                'train_acc': best_train_acc,
                'val_acc': best_val_acc,
                'train_loss': best_train_loss,
                'val_loss': best_val_loss,
                'epochs_run': len(history.history['loss'])
            }])
            row.to_csv(csv_path, mode='a', header=False, index=False)
    
    results_df = pd.read_csv(csv_path)
    return results_df

In [ ]:
# výběr fenotypu
pheno = "PTH"
#pheno = "LPCO_REV_POST"
#pheno = "LSEN"
#pheno = "PEX_REPRO"
#pheno = "APCO_REV_REPRO"

# masky
train_mask_probability = 0.0
test_mask_probability = 0.0

In [ ]:
X = get_snp_matrix(prefix)

In [ ]:
X_ready, y_ready, current_levels, apriori_acc = run_phenotype_pipeline(pheno, X, pheno_path, fam_path)

In [ ]:
# def inspect_top_snps(X, snp_names=None, n_top=10, method='entropy'):
#     """
#     Zobrazí top n_snps podle skóre (entropy nebo variance).
#     """
#     X_f = X.astype(np.float32)

#     if method == 'variance':
#         scores = np.var(X_f, axis=0)
#     elif method == 'entropy':
#         scores = np.zeros(X_f.shape[1], dtype=np.float32)
#         for val in [0, 1, 2]:
#             p = np.mean(X_f == val, axis=0)
#             p = np.clip(p, 1e-10, 1)
#             scores += -p * np.log2(p)

#     top_indices = np.argsort(scores)[::-1][:n_top]

#     rows = []
#     for rank, idx in enumerate(top_indices, 1):
#         counts = {v: int(np.sum(X_f[:, idx] == v)) for v in [0, 1, 2]}
#         total  = X_f.shape[0]
#         rows.append({
#             'rank':    rank,
#             'snp_idx': idx,
#             'name':    snp_names[idx] if snp_names is not None else f'SNP_{idx}',
#             'score':   round(float(scores[idx]), 6),
#             'freq_0':  round(counts[0] / total, 4),
#             'freq_1':  round(counts[1] / total, 4),
#             'freq_2':  round(counts[2] / total, 4),
#             'count_0': counts[0],
#             'count_1': counts[1],
#             'count_2': counts[2],
#         })

#     df = pd.DataFrame(rows).set_index('rank')
#     return df

# df_top = inspect_top_snps(X_ready, snp_names=None, n_top=10, method='entropy')
# print(df_top.to_string())

In [ ]:
pth_lr = [1e-5, 1e-6, 1e-7, 1e-8]
snp_counts = [10, 100, 1000, 10000, 100000, None]
top_method = "entropy"
num_repeats = 10
num_epochs = 50

for lr in pth_lr:
    results = run_snp_scaling_experiment(
        X_ready, y_ready,
        apriori_acc=apriori_acc,
        snp_counts=snp_counts,
        method=top_method,
        n_repeats=num_repeats,
        num_epochs=num_epochs,
        learning_rate=lr,
        csv_path=f"PTH_reduction/test_3/snp_LR{lr}_mask.csv"
    )

In [ ]:
"""
learning_rate = 1e-5
snp_counts = [None]
method = "entropy"
num_epochs = 100

all_indices = select_top_snps(X_ready, n_snps=None, method=method)
all_results = []

for n in snp_counts:
    label = n if n is not None else X_ready.shape[1]
    idx = all_indices if n is None else all_indices[:n]
    X_sub = X_ready[:, idx]

    print(f"\n{'='*50}")
    print(f"SNP: {label}")

    X_train, X_val, y_train, y_val = create_train_val_split(X_sub, y_ready)
    y_integers = np.argmax(y_train, axis=1)
    class_weight_dict = dict(enumerate(compute_class_weight(
        class_weight='balanced', classes=np.unique(y_integers), y=y_integers
    )))

    train_gen = GenomicDataGenerator(X_train, y_train, mask_prob=test_mask_probability)
    val_gen = GenomicDataGenerator(X_val, y_val, mask_prob=test_mask_probability, shuffle=False)

    model = build_genomic_model(X_sub.shape[1] * 4, y_train.shape[1], learning_rate=learning_rate)
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=num_epochs,
        callbacks=[early_stop],
        class_weight=class_weight_dict,
        verbose=1
    )

    best_idx = history.history['val_loss'].index(min(history.history['val_loss']))
    best_val_acc = history.history['val_categorical_accuracy'][best_idx]
    best_val_loss = history.history['val_loss'][best_idx]
    best_train_acc = history.history['categorical_accuracy'][best_idx]
    best_train_loss = history.history['loss'][best_idx]

    print(f"  Train acc: {best_train_acc:.4f} | Val acc: {best_val_acc:.4f} | Apriori: {apriori_acc:.4f}")

    all_results.append({
        'n_snps': label,
        'apriori_acc': apriori_acc,
        'train_acc': best_train_acc,
        'val_acc': best_val_acc,
        'train_loss': best_train_loss,
        'val_loss': best_val_loss,
        'epochs_run': len(history.history['loss'])
    })

results_df = pd.DataFrame(all_results)
"""